# ElasticNet 수출액 예측 — 전체 코드 주피터 노트북

**GEM / v002 / 2026-11-05**

기존에 추가한 네 가지 Python·YAML 파일과 동일한 ElasticNet 핵심 구현을 **노트북 안에도 전부 포함**했습니다. 학습용 데이터 원본은 수정하지 않습니다.

**실행 순서**: 1 환경 확인 → 2 코드 로드 → 3 데이터 계약 검사 → 4 짧은 검증 → 5 전체 학습(직접 설정) → 6 결과 요약 → 7 공통 예측 엔진 검증.

⚠️ `RUN_FULL_TRAIN = True`는 짧은 검증에 성공한 뒤 설정하세요. 전체 학습은 모델이 2개 저장되므로 같은 실행을 무작정 반복하지 마세요.

필수 파일: `data/shared/v002/{train,validation,final_test,manifest}` 및 `configs/{features,splits}.yaml`, `configs/models/elastic_net.yaml`. 2027 예측용 데이터·엔진도 프로젝트에 있어야 공통 엔진을 확인할 수 있습니다.

In [1]:
import sys
from pathlib import Path

ROOT = Path(r"D:\hjs\five-stack")
assert ROOT.is_dir(), f"프로젝트 폴더가 없습니다: {ROOT}"
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("프로젝트:", ROOT)
print("파이썬:", sys.executable)
print("가상환경 확인:", Path(sys.executable).resolve().parent.name.lower() in {"scripts", "bin"})


프로젝트: D:\hjs\five-stack
파이썬: d:\hjs\five-stack\.venv\Scripts\python.exe
가상환경 확인: True


## ElasticNet 전체 구현 코드

아래 셀은 `src/oinq/modeling/supervised/elastic_net.py`의 전체 코드입니다. 해당 `.py` 파일을 수정하지 않으며 노트북에서 동일 함수를 바로 정의합니다.

In [2]:
"""ElasticNet expanding-origin selection, frozen test and deployable pipelines.

Run from the repository root using scripts/train_elastic_net.py.
Only model files are written under artifacts/shared_models; all diagnostics are private.
"""
from __future__ import annotations

import hashlib
import itertools
import json
import re
import warnings
from sklearn.exceptions import ConvergenceWarning
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from oinq.common.config import load_yaml, quarter_number

MODEL_ID = "elastic_net"
TARGET = "target_export_value_usd"
PRED = "predicted_export_value_usd"
KEY = ["country_code", "hs_code", "origin_quarter", "target_quarter"]
VARIANTS = ("with_macro", "without_macro")


def _json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str,
                                     allow_nan=False), encoding="utf-8")


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _require_columns(df, columns, name):
    missing = set(columns).difference(df.columns)
    if missing:
        raise ValueError(f"{name}: missing columns {sorted(missing)}")


def load_inputs(root):
    """Read, but never modify, the released data and team configuration."""
    root = Path(root)
    shared = root / "data/shared/v002"
    manifest = json.loads((shared / "manifest.json").read_text(encoding="utf-8"))
    features_cfg = load_yaml(root / "configs/features.yaml")
    split_cfg = load_yaml(root / "configs/splits.yaml")
    config = load_yaml(root / "configs/models/elastic_net.yaml")
    if manifest.get("dataset_version") != "v002" or not manifest.get("dataset_release_approved"):
        raise ValueError("Expected released and approved dataset v002")
    if split_cfg["validation"]["method"] != "expanding_origin" or split_cfg.get("shuffle"):
        raise ValueError("Only non-shuffled expanding-origin evaluation is supported")
    if manifest["feature_contract"]["target"] != TARGET:
        raise ValueError("Unexpected target")
    features = manifest["feature_columns"]
    fc = manifest["feature_contract"]
    if features != fc["categorical"] + fc["known_future"] + fc["numeric_candidates"]:
        raise ValueError("Feature order does not match manifest")
    if len(set(features)) != len(features) or set(features) & set(fc["forbidden_feature_columns"]):
        raise ValueError("Forbidden or duplicate feature")
    macros = features_cfg["supervised"]["macro"]["columns"]
    removed = [c for c in features if c in macros or c in {f"{m}_missing" for m in macros}]
    if len(macros) != 6 or len(removed) != 12:
        raise ValueError("Expected six macro variables and six missing flags")
    if config.get("model_id") != MODEL_ID or config["training"]["variants"] != list(VARIANTS):
        raise ValueError("ElasticNet YAML model/variant mismatch")
    tables = {}
    manifest_files = {x["path"].replace("\\", "/"): x for x in manifest["files"]}
    for name in ("train", "validation", "final_test"):
        relative = f"data/shared/v002/{name}.parquet"
        path = root / relative
        record = manifest_files.get(relative)
        if record is None or _sha256(path) != record["sha256"]:
            raise ValueError(f"Released checksum mismatch: {relative}")
        df = pd.read_parquet(path)
        _require_columns(df, features + KEY + [TARGET, "target_available", "split", "horizon_quarters"], name)
        if len(df) != manifest["row_counts"][name]["rows"]:
            raise ValueError(f"Unexpected {name} row count")
        if df[KEY].isna().any().any() or df.duplicated(KEY).any():
            raise ValueError(f"Missing or repeated {name} keys")
        if not df["split"].eq(name).all():
            raise ValueError(f"{name} partition labels differ")
        bound = split_cfg["splits"][name]
        q = df.target_quarter.map(quarter_number)
        o = df.origin_quarter.map(quarter_number)
        if not q.between(quarter_number(bound["start"]), quarter_number(bound["end"])).all():
            raise ValueError(f"{name} target quarters outside split")
        if not (q - o).eq(df.horizon_quarters).all():
            raise ValueError(f"{name}: origin/target/horizon inconsistency")
        if not df.horizon_quarters.isin(split_cfg["validation"]["horizon_quarters"]).all():
            raise ValueError(f"{name}: unexpected horizon")
        if not df.target_available.eq(df[TARGET].notna()).all():
            raise ValueError(f"{name}: availability does not match target")
        if (df[TARGET].dropna() < 0).any() or not np.isfinite(df[TARGET].dropna().to_numpy(dtype=float)).all():
            raise ValueError(f"{name}: invalid target")
        for c in ("country_code", "hs_code"):
            if not all(isinstance(x, str) for x in df[c]):
                raise ValueError(f"{name}: {c} must be string")
        df = df.copy()
        df["_target_index"], df["_origin_index"] = q, o
        tables[name] = df
    return tables, manifest, features_cfg, split_cfg, config, removed


def training_rows(pool, origin):
    """Require actual label at or before origin; calendar availability only."""
    selected = pool.loc[pool[TARGET].notna() & pool.target_available.fillna(False)
                        & pool._target_index.le(quarter_number(origin))].copy()
    selected = selected.sort_values(KEY)
    if selected.empty or not selected._origin_index.lt(selected._target_index).all():
        raise ValueError(f"Invalid training pool at {origin}")
    return selected


def prepare_x(df, feature_columns, categorical_columns):
    """Preserve manifest order and safe numeric NaNs; do not fit preprocessing."""
    _require_columns(df, feature_columns, "input")
    x = pd.DataFrame(index=df.index)
    for col in feature_columns:
        if col in categorical_columns:
            x[col] = df[col].astype("string").fillna("__MISSING__").astype(object)
        else:
            x[col] = pd.to_numeric(df[col], errors="raise").to_numpy(dtype=float, na_value=np.nan)
            if np.isinf(x[col].to_numpy(dtype=float)).any():
                raise ValueError(f"Infinite feature: {col}")
    return x


def make_pipeline(feature_columns, categorical_columns, params):
    """Constant imputation also keeps all-missing numeric columns fit-safe."""
    cat = [c for c in feature_columns if c in categorical_columns]
    num = [c for c in feature_columns if c not in categorical_columns]
    if not cat or not num:
        raise ValueError("Categorical and numeric features are both required")
    preprocessor = ColumnTransformer([
        ("cat", Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=0.0,
                                                    keep_empty_features=True)),
                          ("scaler", StandardScaler())]), num),
    ], remainder="drop", sparse_threshold=0.0)
    return Pipeline([("preprocessor", preprocessor),
                     ("model", ElasticNet(**params))])


def metric_values(actual, predicted):
    a = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    valid = np.isfinite(a) & np.isfinite(p)
    a, p = a[valid], p[valid]
    if not len(a):
        return {"n": 0, "excluded_nonfinite": int((~valid).sum()),
                "wape": None, "mae": None, "rmse": None, "smape": None}
    err = np.abs(a - p)
    denom = np.abs(a).sum()
    smape = np.divide(200 * err, np.abs(a) + np.abs(p), out=np.zeros(len(a)),
                      where=(np.abs(a) + np.abs(p)) != 0)
    return {"n": int(len(a)), "excluded_nonfinite": int((~valid).sum()),
            "wape": float(100 * err.sum() / denom) if denom else None,
            "mae": float(err.mean()), "rmse": float(np.sqrt(np.mean((a-p)**2))),
            "smape": float(smape.mean())}


def grouped_metrics(frame, horizons):
    rows = []
    for scope, block in (("h1_5", frame), ("h2_5", frame[frame.horizon_quarters.isin(horizons)])):
        for col, groups in (("overall", [("all", block)]),
                            *[(c, list(block.groupby(c, sort=True))) for c in
                              ("country_code", "hs_code", "horizon_quarters")]):
            for value, section in groups:
                rows.append({"model_id": MODEL_ID, "variant": str(frame.variant.iloc[0]),
                             "scope": scope, "group_by": col, "group_value": str(value),
                             **metric_values(section[TARGET], section[PRED])})
    return pd.DataFrame(rows)


def _predict(pipe, scoring, features, categorical, variant):
    x = prepare_x(scoring, features, categorical)
    raw = np.asarray(pipe.predict(x), dtype=float)
    if not np.isfinite(raw).all():
        raise ValueError("Nonfinite prediction")
    result = scoring[KEY + ["horizon_quarters", TARGET]].copy()
    result["raw_predicted_usd"] = raw
    result[PRED] = np.maximum(raw, 0)
    result["negative_raw_prediction"] = raw < 0
    result["variant"] = variant
    result["model_id"] = MODEL_ID
    return result


def _fit_and_predict(fit, scoring, features, cats, params, variant):
    model = make_pipeline(features, cats, params)
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(prepare_x(fit, features, cats), fit[TARGET].to_numpy(dtype=float))
    model._convergence_warnings = sum(issubclass(w.category, ConvergenceWarning) for w in recorded)
    return model, _predict(model, scoring, features, cats, variant)


def _params(config):
    base = {k: config["params"][k] for k in ("max_iter", "tol", "selection", "random_state")
            if k in config["params"]}
    candidates = [{**base, "alpha": float(a), "l1_ratio": float(r)}
                  for a, r in itertools.product(config["search"]["alpha"], config["search"]["l1_ratio"])]
    if not candidates or any(p["alpha"] <= 0 or not 0 <= p["l1_ratio"] <= 1 for p in candidates):
        raise ValueError("Invalid ElasticNet parameter grid")
    return candidates


def _make_run_dir(root, run_id):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", run_id):
        raise ValueError("Unsafe run ID")
    path = root / ".private/elastic_net" / run_id
    path.mkdir(parents=True, exist_ok=False)
    return path


def _save_bundle(path, pipe, features, cats, trained_quarter, x_check):
    if path.exists():
        raise FileExistsError(f"Shared model already exists: {path}")
    bundle = {"pipeline": pipe, "feature_columns": list(features),
              "categorical_columns": list(cats), "dataset_version": "v002",
              "trained_through_target_quarter": trained_quarter}
    before = np.asarray(pipe.predict(x_check), dtype=float)
    if not np.isfinite(before).all():
        raise ValueError("Nonfinite saved-model check predictions")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        joblib.dump(bundle, path, compress=3)
        restored = joblib.load(path)
        if restored["feature_columns"] != features or restored["categorical_columns"] != cats:
            raise AssertionError("Reloaded feature metadata differs")
        after = np.asarray(restored["pipeline"].predict(x_check), dtype=float)
        np.testing.assert_array_equal(before, after)
    except Exception:
        path.unlink(missing_ok=True)
        raise
    return True


def run_training(root, run_id=None, max_origins=None, publish=True):
    root = Path(root).resolve()
    tables, manifest, feature_cfg, splits, config, removed = load_inputs(root)
    run_id = run_id or "elastic_net_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    private = _make_run_dir(root, run_id)
    metadata = {"run_id": run_id, "model_id": MODEL_ID, "dataset_version": "v002",
                "historical_asof_verified": False,
                "note": "Calendar-only label availability; publication-at-origin not independently verified.",
                "selection_split": "validation", "final_test_for_tuning": False,
                "deployment_uses_latest_actuals_after_frozen_test": True,
                "published": False}
    _json(private / "run_metadata.json", metadata)
    train, validation, test = (tables[k] for k in ("train", "validation", "final_test"))
    pool = pd.concat([train, validation], ignore_index=True)
    categorical = manifest["feature_contract"]["categorical"]
    primary = splits["validation"]["primary_horizons"]
    eligible_val = validation[TARGET].notna() & validation.target_available.fillna(False)
    val = validation.loc[eligible_val].copy()
    origins = sorted(val.origin_quarter.unique(), key=quarter_number)
    if max_origins is not None:
        if max_origins < 1:
            raise ValueError("max_origins must be positive")
        origins = origins[:max_origins]
        val = val[val.origin_quarter.isin(origins)].copy()
        publish = False  # smoke test cannot create a production model
    # Select on the same eligible rows/origins for all candidate parameters.
    if val.empty:
        raise ValueError("No validation rows")
    candidates = _params(config)
    search_rows, winning = [], {}
    for variant in VARIANTS:
        features = [c for c in manifest["feature_columns"] if variant == "with_macro" or c not in removed]
        trial_results = []
        for i, params in enumerate(candidates):
            portions = []
            convergence_warnings = 0
            for origin in origins:
                fit = training_rows(pool, origin)
                score = val[val.origin_quarter.eq(origin)]
                model, predicted = _fit_and_predict(fit, score, features, categorical, params, variant)
                portions.append(predicted)
                convergence_warnings += model._convergence_warnings
                print(f"[{variant}] candidate {i+1}/{len(candidates)}, origin {origin}: "
                      f"fit={len(fit)}, score={len(score)}", flush=True)
            predictions = pd.concat(portions, ignore_index=True)
            primary_rows = predictions[predictions.horizon_quarters.isin(primary)]
            score = metric_values(primary_rows[TARGET], primary_rows[PRED])
            if score["n"] != len(primary_rows) or score["wape"] is None:
                raise ValueError("Undefined or incomplete primary validation metric")
            record = {"variant": variant, "candidate": i, **params, **score,
                      "convergence_warnings": convergence_warnings,
                      "negative_raw_predictions": int(predictions.negative_raw_prediction.sum())}
            search_rows.append(record)
            trial_results.append((convergence_warnings, score["wape"], i, params, predictions))
        converged = [r for r in trial_results if r[0] == 0]
        if not converged:
            raise ValueError(f"All {variant} parameter candidates had convergence warnings; "
                             "increase max_iter/tol or revise search grid in elastic_net.yaml")
        _, _, _, best_params, best_predictions = min(converged, key=lambda v: (v[1], v[2]))
        winning[variant] = (features, best_params, best_predictions)
    pd.DataFrame(search_rows).to_csv(private / "parameter_search.csv", index=False)
    all_val = []
    for variant, (features, params, predicted) in winning.items():
        all_val.append(predicted)
        grouped_metrics(predicted, primary).to_csv(private / f"validation_metrics_{variant}.csv",
                                                   index=False, encoding="utf-8-sig")
    pd.concat(all_val, ignore_index=True).to_parquet(private / "validation_predictions.parquet", index=False)
    # The smoke path stops before final test or publishing.
    if not publish:
        metadata.update(status="validation_only", selected_params={v: p for v, (_, p, _) in winning.items()})
        _json(private / "run_metadata.json", metadata)
        return private
    test_origin = splits["final_test"]["origin_quarter"]
    if test_origin != "2025Q4":
        raise ValueError("Unexpected frozen test origin")
    eligible_test = (test.origin_quarter.eq(test_origin) & test[TARGET].notna()
                     & test.target_available.fillna(False)
                     & test.target_quarter.isin(splits["final_test"]["target_quarters"]))
    test_rows = test.loc[eligible_test].copy()
    if test_rows.empty:
        raise ValueError("No frozen test labels")
    fit_test = training_rows(pool, test_origin)
    all_test, test_summaries = [], {}
    for variant, (features, params, _) in winning.items():
        pipe, predicted = _fit_and_predict(fit_test, test_rows, features, categorical, params, variant)
        if pipe._convergence_warnings:
            raise ValueError(f"Frozen test fit did not converge: {variant}")
        all_test.append(predicted)
        grouped_metrics(predicted, primary).to_csv(private / f"final_test_metrics_{variant}.csv",
                                                   index=False, encoding="utf-8-sig")
        test_summaries[variant] = metric_values(predicted[TARGET], predicted[PRED]) | {
            "negative_raw_predictions": int(predicted.negative_raw_prediction.sum())}
    pd.concat(all_test, ignore_index=True).to_parquet(private / "final_test_predictions.parquet", index=False)
    # Only AFTER recording the frozen final-test results, fit through latest actuals.
    latest_pool = pd.concat([pool, test], ignore_index=True)
    fit_deploy = training_rows(latest_pool, "2026Q3")
    trained_through = max(fit_deploy.target_quarter.unique(), key=quarter_number)
    if trained_through != "2026Q3":
        raise ValueError("Deployment labels do not reach 2026Q3")
    files = {}
    # Avoid half-publishing when the other shared model path already exists.
    for variant in VARIANTS:
        p = root / "artifacts/shared_models" / MODEL_ID / variant / "pipeline.joblib"
        if p.exists():
            raise FileExistsError(f"Existing shared model is protected: {p}")
        files[variant] = p
    created = []
    try:
        for variant, (features, params, _) in winning.items():
            pipe = make_pipeline(features, categorical, params)
            with warnings.catch_warnings(record=True) as recorded:
                warnings.simplefilter("always", ConvergenceWarning)
                pipe.fit(prepare_x(fit_deploy, features, categorical),
                         fit_deploy[TARGET].to_numpy(dtype=float))
            if any(issubclass(w.category, ConvergenceWarning) for w in recorded):
                raise ValueError(f"Deployment fit did not converge: {variant}")
            x_check = prepare_x(test_rows.head(12), features, categorical)
            _save_bundle(files[variant], pipe, features, categorical, trained_through, x_check)
            created.append(files[variant])
    except Exception:
        for path in created:
            path.unlink(missing_ok=True)
        raise
    metadata.update(status="completed", published=True, saved_paths={v: str(p) for v, p in files.items()},
                    selected_params={v: p for v, (_, p, _) in winning.items()},
                    validation_primary={v: metric_values(pred[pred.horizon_quarters.isin(primary)][TARGET],
                                                         pred[pred.horizon_quarters.isin(primary)][PRED]) | {
                        "negative_raw_predictions": int(pred.negative_raw_prediction.sum())}
                                        for v, (_, _, pred) in winning.items()},
                    frozen_test=test_summaries, trained_through_target_quarter=trained_through,
                    pipeline_reload_exact=True)
    _json(private / "run_metadata.json", metadata)
    print(json.dumps({"validation_primary": metadata["validation_primary"],
                      "frozen_test": test_summaries, "saved_paths": metadata["saved_paths"]},
                     ensure_ascii=False, indent=2), flush=True)
    return private


## 1. 데이터 및 설정 확인

원본을 읽고 계약을 검증하기만 합니다. 값이나 파일을 고치지 않습니다.

In [3]:
tables, manifest, feature_cfg, split_cfg, model_cfg, macro_removed = load_inputs(ROOT)
print("버전:", manifest["dataset_version"])
print("전체 입력 변수:", len(manifest["feature_columns"]))
print("제외할 거시 관련 변수:", len(macro_removed), macro_removed)
for name, df in tables.items():
    print(name, df.shape, "정답 존재:", int(df[TARGET].notna().sum()))


버전: v002
전체 입력 변수: 57
제외할 거시 관련 변수: 12 ['gdp_usd', 'gdp_per_capita_usd', 'population', 'inflation_pct', 'exchange_rate_lcu_per_usd', 'internet_users_pct', 'gdp_usd_missing', 'gdp_per_capita_usd_missing', 'population_missing', 'inflation_pct_missing', 'exchange_rate_lcu_per_usd_missing', 'internet_users_pct_missing']
train (28800, 89) 정답 존재: 23290
validation (12800, 89) 정답 존재: 11455
final_test (480, 89) 정답 존재: 447


## 2. 간단한 학습 검증 (2개 기준시점)

모델은 공유 폴더에 저장하지 않고 `.private/elastic_net/`에만 진단 결과를 저장합니다. 파라미터 후보 수에 따라 실행 시간이 걸릴 수 있습니다.

In [4]:
RUN_SMOKE = False  # 이미 검증 완료. 다시 시험할 때만 True
if RUN_SMOKE:
    smoke_output = run_training(ROOT, max_origins=2, publish=False)
    print("간단 검증 폴더:", smoke_output)
else:
    print("간단 검증 건너뜀 (기존 결과 유지)")

간단 검증 건너뜀 (기존 결과 유지)


## 3. 전체 학습 (이미 완료 — 기본 실행 안 함)

2026-11-05 전체 학습과 모델 두 개 저장을 마쳤으므로 `RUN_FULL_TRAIN=False`로 두었습니다. **Run All을 눌러도 다시 학습하거나 기존 공유 모델을 덮어쓰지 않습니다.** 재학습은 팀장과 조율 후 실시하세요.

In [5]:
RUN_FULL_TRAIN = False  # 전체 학습·저장 완료. 기존 모델 보호를 위해 False 유지
if RUN_FULL_TRAIN:
    training_output = run_training(ROOT, publish=True)
    print("전체 학습 결과 폴더:", training_output)
else:
    print("전체 학습 건너뜀: 저장된 모델과 평가 보고서를 재사용합니다.")

전체 학습 건너뜀: 저장된 모델과 평가 보고서를 재사용합니다.


## 4. 최신 보고서 확인

모델 평가 지표는 실제 전체 학습을 완료한 후에만 확인할 수 있습니다.

In [6]:
import json
private_root = ROOT / ".private" / "elastic_net"
reports = sorted(private_root.glob("*/run_metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True) if private_root.exists() else []
if reports:
    report = json.loads(reports[0].read_text(encoding="utf-8"))
    print("최신 보고서:", reports[0])
    print("상태:", report.get("status"))
    for section in ("selected_params", "validation_primary", "frozen_test", "saved_paths"):
        if section in report:
            print("\n", section, json.dumps(report[section], ensure_ascii=False, indent=2))
else:
    print("저장된 학습 보고서가 아직 없습니다.")


최신 보고서: D:\hjs\five-stack\.private\elastic_net\elastic_net_20261105T071327Z\run_metadata.json
상태: completed

 selected_params {
  "with_macro": {
    "max_iter": 5000,
    "tol": 0.0001,
    "selection": "cyclic",
    "random_state": 42,
    "alpha": 10000.0,
    "l1_ratio": 0.8
  },
  "without_macro": {
    "max_iter": 5000,
    "tol": 0.0001,
    "selection": "cyclic",
    "random_state": 42,
    "alpha": 10000.0,
    "l1_ratio": 0.8
  }
}

 validation_primary {
  "with_macro": {
    "n": 9164,
    "excluded_nonfinite": 0,
    "wape": 124.94456242113176,
    "mae": 13469583.651535919,
    "rmse": 54434348.52120337,
    "smape": 148.82911803989194,
    "negative_raw_predictions": 0
  },
  "without_macro": {
    "n": 9164,
    "excluded_nonfinite": 0,
    "wape": 124.94967940380815,
    "mae": 13470135.28519542,
    "rmse": 54438135.82432101,
    "smape": 148.8411201557389,
    "negative_raw_predictions": 0
  }
}

 frozen_test {
  "with_macro": {
    "n": 447,
    "excluded_nonfinite":

## 5. 저장 모델 재로딩 검사

저장된 모델과 전처리기를 다시 학습하지 않고 예측합니다.

In [7]:
import joblib
import numpy as np
for variant in VARIANTS:
    path = ROOT / "artifacts" / "shared_models" / "elastic_net" / variant / "pipeline.joblib"
    if not path.exists():
        print(variant, "모델 파일 없음 (전체 학습 후 생성됨)")
        continue
    bundle = joblib.load(path)
    assert bundle["dataset_version"] == "v002"
    check_rows = tables["final_test"].head(10)
    X = prepare_x(check_rows, bundle["feature_columns"], bundle["categorical_columns"])
    a = bundle["pipeline"].predict(X)
    b = joblib.load(path)["pipeline"].predict(X)
    np.testing.assert_array_equal(a, b)
    print(variant, "재로딩 결과 일치:", len(a), "행; 학습 정답 기준:", bundle["trained_through_target_quarter"])


with_macro 재로딩 결과 일치: 10 행; 학습 정답 기준: 2026Q3
without_macro 재로딩 결과 일치: 10 행; 학습 정답 기준: 2026Q3


## 6. 공통 예측 엔진 실행

팀장의 `scripts/forecast.py`를 사용합니다. 프로젝트의 공통 엔진을 임의 수정하지 않습니다. 결과의 640행, 누락 및 중복을 확인하려면 실행 결과 폴더도 확인하세요.

In [8]:
import subprocess
RUN_FORECAST = False  # 두 개의 공유 모델이 저장된 뒤 True로 바꾸세요.
if RUN_FORECAST:
    for variant in VARIANTS:
        command = [sys.executable, str(ROOT / "scripts" / "forecast.py"), "--model", "elastic_net", "--variant", variant]
        print("실행:", " ".join(command))
        result = subprocess.run(command, cwd=str(ROOT), capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"공통 엔진 연결 실패: {variant} (공통 코드를 수정하지 말고 오류를 공유하세요)")
else:
    print("공통 엔진 미실행: RUN_FORECAST를 True로 바꿔주세요.")


공통 엔진 미실행: RUN_FORECAST를 True로 바꿔주세요.


## 7. 완료된 학습 보고서 요약 (재학습 없음)

저장된 최신 `run_metadata.json`에서 검증·최종 테스트 점수를 읽습니다. `final_test`가 아니라 **`frozen_test`** 키를 사용합니다.

In [9]:
import json
from pathlib import Path

REPORT_PATH = ROOT / ".private" / "elastic_net" / "elastic_net_20261105T071327Z" / "run_metadata.json"
if not REPORT_PATH.is_file():
    raise FileNotFoundError(f"학습 보고서가 없습니다: {REPORT_PATH}")
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print("보고서:", REPORT_PATH)
print("상태:", report.get("status"))
print("모델 재로딩 일치:", report.get("pipeline_reload_exact"))
print("학습 정답 마지막 분기:", report.get("trained_through_target_quarter"))
print("\n선택 파라미터:")
print(json.dumps(report.get("selected_params", {}), ensure_ascii=False, indent=2))

보고서: D:\hjs\five-stack\.private\elastic_net\elastic_net_20261105T071327Z\run_metadata.json
상태: completed
모델 재로딩 일치: True
학습 정답 마지막 분기: 2026Q3

선택 파라미터:
{
  "with_macro": {
    "max_iter": 5000,
    "tol": 0.0001,
    "selection": "cyclic",
    "random_state": 42,
    "alpha": 10000.0,
    "l1_ratio": 0.8
  },
  "without_macro": {
    "max_iter": 5000,
    "tol": 0.0001,
    "selection": "cyclic",
    "random_state": 42,
    "alpha": 10000.0,
    "l1_ratio": 0.8
  }
}


In [10]:
import pandas as pd
rows = []
for variant in ("with_macro", "without_macro"):
    for section, label in (("validation_primary", "검증 h2~5"), ("frozen_test", "최종 테스트")):
        result = report.get(section, {}).get(variant, {})
        rows.append({"구성": variant, "구분": label,
                     **{name: result.get(name) for name in ("n", "wape", "mae", "rmse", "smape", "negative_raw_predictions", "excluded_nonfinite")}})
metrics_table = pd.DataFrame(rows)
display(metrics_table)
assert report.get("status") == "completed", "보고서가 completed 상태가 아닙니다."
assert report.get("pipeline_reload_exact") is True, "모델 재로딩 일치 기록을 확인하세요."
assert all(pd.notna(metrics_table["wape"])), "일부 성능 지표가 누락됐습니다."

,구성,구분,n,wape,mae,rmse,smape,negative_raw_predictions,excluded_nonfinite
0,with_macro,검증 h2~5,9164,124.944562,1.346958e+07,5.443435e+07,148.829118,0,0
1,with_macro,최종 테스트,447,118.233629,1.692842e+07,5.975307e+07,143.394038,0,0
2,without_macro,검증 h2~5,9164,124.949679,1.347014e+07,5.443814e+07,148.841120,0,0
3,without_macro,최종 테스트,447,118.241341,1.692953e+07,5.975738e+07,143.398917,0,0


## 8. 2027년 공통 예측 엔진 실행 및 결과 검사

아래 셀은 **저장된 모델만** 사용하여 `scripts/forecast.py`를 각 구성별로 실행합니다. 팀 공통 코드는 수정하지 않습니다. 실행 후 `.private/forecasts/`에 생성된 결과에서 행 수·조합·결측·무한대 여부를 점검합니다. **이 섹션만 실행**하면 재학습되지 않습니다.

In [11]:
import subprocess
import sys
from pathlib import Path
import pandas as pd
import numpy as np

FORECAST_RUNS = {}
for variant in ("with_macro", "without_macro"):
    print("\n=====", variant, "===== ")
    cmd = [sys.executable, str(ROOT / "scripts" / "forecast.py"),
           "--model", "elastic_net", "--variant", variant]
    result = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True)
    print("종료 코드:", result.returncode)
    print(result.stdout)
    if result.stderr:
        print("표준 오류 출력:", result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"공통 예측 엔진 실패: {variant}. 공통 엔진 코드를 수정하지 말고 팀장에게 오류를 전달하세요.")
    # 실행 직후 생성된 폴더를 찾아 두 구성 결과를 분리해 저장
    forecast_root = ROOT / ".private" / "forecasts"
    matching = []
    for folder in forecast_root.iterdir() if forecast_root.is_dir() else []:
        meta_path = folder / "run_metadata.json"
        if meta_path.is_file():
            try:
                info = json.loads(meta_path.read_text(encoding="utf-8"))
                if info.get("model_id") == "elastic_net" and info.get("model_variant") == variant:
                    matching.append((meta_path.stat().st_mtime, folder, info))
            except (ValueError, OSError):
                pass
    if matching:
        _, folder, info = max(matching, key=lambda x:x[0])
        FORECAST_RUNS[variant] = (folder, info)
        print("결과 폴더:", folder)
        print("분기 행 수:", info.get("quarterly_rows"), "/ 연간 행 수:", info.get("annual_rows"))
        assert info.get("quarterly_rows") == 640, f"{variant}: 분기별 640행이 아닙니다."
        assert info.get("annual_rows") == 160, f"{variant}: 연간 160행이 아닙니다."
    else:
        print("경고: 생성된 결과 메타데이터를 자동으로 찾지 못했습니다. 위 출력의 저장 경로를 확인하세요.")


===== with_macro ===== 
종료 코드: 0
완료: elastic_net/with_macro | 2026Q3 → 2027Q1, 2027Q2, 2027Q3, 2027Q4
분기 640행 / 연간 160행 / 예측 640행 / 입력 결측 포함 640행
저장: D:\hjs\five-stack\.private\forecasts\forecast_20261105T072608436236Z (재학습 없음)

경고: 생성된 결과 메타데이터를 자동으로 찾지 못했습니다. 위 출력의 저장 경로를 확인하세요.

===== without_macro ===== 
종료 코드: 0
완료: elastic_net/without_macro | 2026Q3 → 2027Q1, 2027Q2, 2027Q3, 2027Q4
분기 640행 / 연간 160행 / 예측 640행 / 입력 결측 포함 612행
저장: D:\hjs\five-stack\.private\forecasts\forecast_20261105T072612301560Z (재학습 없음)

경고: 생성된 결과 메타데이터를 자동으로 찾지 못했습니다. 위 출력의 저장 경로를 확인하세요.


In [12]:
# 실행 결과의 구조·조합 검사 (공통 엔진 저장 파일명을 임의 추측하지 않음)
import json
import pandas as pd
import numpy as np

for variant, (folder, info) in FORECAST_RUNS.items():
    print(f"\n[{variant}]", folder)
    print("메타데이터: ", {k:info.get(k) for k in ("quarterly_rows", "annual_rows", "predicted_rows", "origin_quarter", "target_quarters")})
    # 정확히 분기 예측 640행을 담은 파일을 찾고, 테이블에 필요한 컬럼이 있는지 확인
    files = list(folder.rglob("*.parquet")) + list(folder.rglob("*.csv"))
    candidates = []
    for p in files:
        try:
            df = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
            if len(df) == 640 and {"country_code", "hs_code", "target_quarter"}.issubset(df.columns):
                candidates.append((p, df))
        except Exception as ex:
            print("읽기 건너뜀:", p.name, type(ex).__name__)
    if not candidates:
        print("640행 예측표를 자동으로 찾지 못했습니다. 결과 폴더 내 파일명·컬럼을 확인하세요.")
        continue
    p, df = candidates[0]
    print("검사 파일:", p.name)
    print("행 수:", len(df))
    assert len(df) == 640
    assert df["country_code"].nunique() == 20
    assert df["hs_code"].nunique() == 8
    assert df["target_quarter"].nunique() == 4
    assert not df.duplicated(["country_code", "hs_code", "target_quarter"]).any(), "조합 중복 있음"
    value_cols = [c for c in df.columns if c in ("predicted_export_value_usd", "forecast_export_value_usd", "prediction_usd")]
    if not value_cols:
        print("예측 금액 열 이름 확인 필요:", list(df.columns))
    else:
        col = value_cols[0]
        values = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)
        assert np.isfinite(values).all(), "예측 금액 결측/무한대 발견"
        assert (values >= 0).all(), "최종 예측 금액 음수 발견"
        print("예측 금액 결측·무한대·음수 없음:", col)
    print("조합 검사 통과")

## 9. 담당 테스트 실행 (선택)

`tests/test_elastic_net_training.py`가 프로젝트에 있는 경우 pytest를 실행합니다. 학습·모델 공유 파일을 다시 만들지 않습니다.

In [14]:
import subprocess
import sys
TEST_PATH = ROOT / "tests" / "test_elastic_net_training.py"
if TEST_PATH.exists():
    result = subprocess.run([sys.executable, "-m", "pytest", str(TEST_PATH), "-q"],
                            cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("테스트 종료 코드:", result.returncode)
else:
    print("테스트 파일 없음:", TEST_PATH)

.....                                                                    [100%]
5 passed in 3.83s

테스트 종료 코드: 0


## 10. 제출 전 확인

`artifacts/shared_models`에는 두 `pipeline.joblib`만 공유하세요. `.private/`의 보고서·CSV·로그와 이 실험 노트북은 팀장님 요청에 따라 Git 제출 대상에서 제외합니다. 담당 코드 4개와 모델 2개만 선별 커밋하고, main 병합은 팀장님과 조율하세요.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"D:\hjs\five-stack")

runs = {
    "with_macro": "forecast_20261105T072608436236Z",
    "without_macro": "forecast_20261105T072612301560Z",
}

KEY = ["country_code", "hs_code", "target_quarter"]

for variant, run_id in runs.items():
    path = ROOT / ".private" / "forecasts" / run_id
    df = pd.read_parquet(path / "quarterly_forecast.parquet")

    print(f"\n===== {variant} =====")
    print("전체 행:", len(df))
    print("중복 조합:", df.duplicated(KEY).sum())
    print("국가 수:", df["country_code"].nunique())
    print("품목 수:", df["hs_code"].nunique())
    print("예측 분기 수:", df["target_quarter"].nunique())

    print("\n예측 관련 컬럼:")
    print([c for c in df.columns if "predict" in c or "forecast" in c])


===== with_macro =====
전체 행: 640
중복 조합: 0
국가 수: 20
품목 수: 8
예측 분기 수: 4

예측 관련 컬럼:
['raw_predicted_export_value_usd', 'predicted_export_value_usd', 'prediction_status']

===== without_macro =====
전체 행: 640
중복 조합: 0
국가 수: 20
품목 수: 8
예측 분기 수: 4

예측 관련 컬럼:
['raw_predicted_export_value_usd', 'predicted_export_value_usd', 'prediction_status']


In [16]:
for variant, run_id in runs.items():
    path = ROOT / ".private" / "forecasts" / run_id
    df = pd.read_parquet(path / "quarterly_forecast.parquet")

    raw = pd.to_numeric(
        df["raw_predicted_export_value_usd"], errors="coerce"
    )
    pred = pd.to_numeric(
        df["predicted_export_value_usd"], errors="coerce"
    )

    expected = pd.MultiIndex.from_product(
        [
            manifest["country_codes"],
            manifest["hs_codes"],
            ["2027Q1", "2027Q2", "2027Q3", "2027Q4"]
        ],
        names=["country_code", "hs_code", "target_quarter"]
    )

    actual = pd.MultiIndex.from_frame(
        df[["country_code", "hs_code", "target_quarter"]]
    )

    print(f"\n===== {variant} =====")
    print("누락 조합:", len(expected.difference(actual)))
    print("예상 외 조합:", len(actual.difference(expected)))
    print("중복 조합:", actual.duplicated().sum())
    print("예측 금액 결측:", pred.isna().sum())
    print("예측 금액 무한대:", np.isinf(pred).sum())
    print("최종 음수 예측:", (pred < 0).sum())
    print("원예측 음수:", (raw < 0).sum())
    print("원예측 결측:", raw.isna().sum())
    print("원예측 무한대:", np.isinf(raw).sum())
    print("음수 제한 규칙 일치:",
          np.allclose(pred, np.maximum(raw, 0)))


===== with_macro =====
누락 조합: 0
예상 외 조합: 0
중복 조합: 0
예측 금액 결측: 0
예측 금액 무한대: 0
최종 음수 예측: 0
원예측 음수: 0
원예측 결측: 0
원예측 무한대: 0
음수 제한 규칙 일치: True

===== without_macro =====
누락 조합: 0
예상 외 조합: 0
중복 조합: 0
예측 금액 결측: 0
예측 금액 무한대: 0
최종 음수 예측: 0
원예측 음수: 0
원예측 결측: 0
원예측 무한대: 0
음수 제한 규칙 일치: True
